# 🤖 AI Engineering Fundamentals — Lezione 3
## Notebook Gruppo C

**ITS Novitas 4.0 | Martedì 26/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "C"
MEMBRI = ["", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo C: Memoria Persistente & Streaming

Costruite un chatbot che ricorda tra sessioni diverse
e risponde in tempo reale con lo streaming.

---
### Esercizio 1 — Memoria persistente su JSON *(guidato)*

Salvate la history su file JSON e ricaricatela.
Simulate due sessioni separate — il chatbot deve ricordare
quello che avete detto nella sessione precedente.

In [ ]:
# Esercizio 1 — memoria persistente

MEMORY_FILE = "chat_widata.json"

def carica_storia():
    """Carica la history dal file. Restituisce lista vuota se non esiste."""
    # TODO: se MEMORY_FILE esiste, aprilo e restituisci il contenuto
    # altrimenti restituisci []
    # Attenzione: usa encoding="utf-8"
    ___

def salva_storia(history):
    """Salva la history su file JSON."""
    # TODO: salva history su MEMORY_FILE
    # Attenzione: usa ensure_ascii=False e indent=2
    ___

def chat_persistente(messaggio, history):
    history.append({"role": "user", "content": messaggio})
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    salva_storia(history)  # salva dopo ogni messaggio
    return testo

# ── SESSIONE 1 ────────────────────────────────────────────────────
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()
print(f"Messaggi precedenti caricati: {len(history)}")

r = chat_persistente("Ciao! Mi chiamo Sara e sto imparando AI Engineering.", history)
print(f"🤖 {r}\n")
r = chat_persistente("Il mio progetto finale è un chatbot per WiData Srl.", history)
print(f"🤖 {r}\n")

print(f"History salvata: {len(history)} messaggi\n")

# ── SESSIONE 2 — nuova sessione, stessa memoria ───────────────────
print("=" * 50)
print("SESSIONE 2 (nuova sessione)")
print("=" * 50)

history = carica_storia()  # ricarica
print(f"Messaggi precedenti caricati: {len(history)}")

r = chat_persistente("Come mi chiamo e su cosa sto lavorando?", history)
print(f"🤖 {r}")

# Il chatbot ricorda? Se sì, perché? Se no, cosa manca?
# ...

---
### Esercizio 2 — Streaming *(guidato)*

Implementate lo streaming e confrontate la differenza percepita
rispetto alla chiamata normale. Testate con e senza `flush=True`.

In [ ]:
# Esercizio 2 — streaming
import time

domanda_lunga = "Spiegami in dettaglio come funziona un sistema IoT completo, dalla raccolta dei dati dei sensori fino alla visualizzazione sulla piattaforma cloud."

# ── SENZA streaming ───────────────────────────────────────────────
print("SENZA streaming:")
t_start = time.time()
risposta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": domanda_lunga}]
)
t_fine = time.time()
print(risposta.content[0].text[:100]+"...")
print(f"⏱ Tempo prima parola: {t_fine-t_start:.2f}s (tutto insieme)\n")

# ── CON streaming ─────────────────────────────────────────────────
print("CON streaming (con flush=True):")
t_start = time.time()
primo_token = True

# TODO: usate client.messages.stream() con un context manager
# Per ogni text in stream.text_stream:
#   - se è il primo token, stampate il tempo trascorso
#   - stampate il testo con end="" e flush=True
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": domanda_lunga}]
) as stream:
    for text in stream.text_stream:
        if primo_token:
            print(f"⏱ Primo token: {time.time()-t_start:.2f}s")
            primo_token = False
        # TODO: stampate il testo con flush=True
        ___

print("\n")
print("💡 Ora togliete flush=True e rieseguite. Cosa cambia?")

---
### Esercizio 3 — Chatbot completo: memoria + streaming *(libero)*

Combinate tutto: history persistente su JSON + streaming.
Il chatbot deve:
- Ricordare tra sessioni diverse
- Rispondere in streaming
- Avere un system prompt WiData
- Salvare dopo ogni messaggio

In [ ]:
# Esercizio 3 — chatbot completo

MEMORY_FILE_V2 = "chat_widata_v2.json"

SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, startup IoT di Sassari.
Rispondi in modo professionale su prodotti e servizi IoT.
Ricorda le informazioni che l'utente ti fornisce durante la conversazione.
"""

def chat_completo(messaggio, history):
    """Chatbot con memoria persistente + streaming."""
    history.append({"role": "user", "content": messaggio})

    print("🤖 ", end="", flush=True)
    testo_completo = ""

    # TODO: implementate lo streaming con system=SYSTEM_WIDATA
    # Accumulate il testo in testo_completo
    # Aggiungete la risposta alla history
    # Salvate la history su file
    ___

    print()  # newline dopo lo streaming
    return testo_completo

# Test: sessione 1
history = carica_storia()
chat_completo("Ciao! Sto valutando i vostri sensori per un progetto smart city.", history)
chat_completo("Ho un budget di €5000 per 50 punti di monitoraggio.", history)

# Riavviate il kernel e rieseguite solo la sessione 2:
# history = carica_storia()
# chat_completo("Ricordi il mio budget e quanti punti devo monitorare?", history)

---
### Esercizio 4 — Gestione errori e robustezza *(libero)*

Cosa succede se il file JSON è corrotto?
E se l'utente interrompe lo streaming a metà?

Rendete il chatbot robusto aggiungendo gestione degli errori.

In [ ]:
# Esercizio 4 — robustezza

# Caso 1: file JSON corrotto
# Simulate un file corrotto e verificate che carica_storia() gestisca l'errore
with open("chat_corrotto.json", "w") as f:
    f.write("{questo non è json valido")

def carica_storia_robusta(filepath):
    """Versione robusta che gestisce file corrotti."""
    # TODO: aggiungete try/except per json.JSONDecodeError
    # Se il file è corrotto, loggare l'errore e restituire []
    ___

history_test = carica_storia_robusta("chat_corrotto.json")
print(f"Caricamento file corrotto: {len(history_test)} messaggi (atteso: 0)")

# Caso 2: streaming interrotto
# Cosa succede alla history se lo streaming viene interrotto?
# Come gestite il messaggio parziale?

def chat_streaming_robusto(messaggio, history):
    """Streaming con gestione dell'interruzione."""
    history.append({"role": "user", "content": messaggio})
    testo_parziale = ""
    try:
        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=history
        ) as stream:
            for text in stream.text_stream:
                testo_parziale += text
                print(text, end="", flush=True)
    except KeyboardInterrupt:
        # TODO: se l'utente interrompe, salvate comunque il testo parziale
        print("\n⚠️ Streaming interrotto")
        ___
    finally:
        print()

# Conclusione:
# Quali errori avete trovato?
# Come li avete risolti?
# ...

---
## 📊 Preparate la presentazione (5 slide)

1. **Memoria persistente** — come funziona, demo chiudi-riapri-ricorda
2. **ensure_ascii=False** — mostrate cosa succede senza (con accenti italiani)
3. **Streaming** — il confronto tempo primo token con e senza
4. **flush=True** — mostrate cosa succede senza
5. **Il chatbot completo** — demo live del chatbot con memoria + streaming

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*